# 03 — Cohort definition

This is where I narrow the ~100 demo stays into the analytic cohort I'll actually model on. The point of doing this in a separate notebook is that the inclusion/exclusion criteria are decisions that deserve to be explicit and replayable — both for me reading my own code later, and for anyone trying to reproduce.

## Criteria

1. **Adults**: `anchor_age >= 18`.
2. **First ICU stay per patient**: avoids correlation between rows of the same patient.
3. **Length of stay >= 24h**: I'm predicting from the first 24h of the stay, so anyone who left or died before 24h can't have a complete feature window. This is a standard simplification; it biases the cohort against the earliest deaths, which I note here and revisit later.
4. **No missing key fields** for the outcome and core demographics.

The resulting cohort is saved to disk so the feature and model notebooks can start from a clean snapshot.

In [8]:
from pathlib import Path

import duckdb
import pandas as pd

DATA_DIR = Path("../data")
HOSP = DATA_DIR / "hosp"
ICU = DATA_DIR / "icu"
DERIVED = DATA_DIR / "derived"
DERIVED.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()

## Load the full stay-level table

In [9]:
query = f"""
SELECT
    i.subject_id,
    i.hadm_id,
    i.stay_id,
    i.intime,
    i.outtime,
    i.los,
    i.first_careunit,
    p.gender,
    p.anchor_age,
    a.race,
    a.insurance,
    a.marital_status,
    a.hospital_expire_flag
FROM '{ICU / 'icustays.csv.gz'}' AS i
LEFT JOIN '{HOSP / 'patients.csv.gz'}' AS p
    ON i.subject_id = p.subject_id
LEFT JOIN '{HOSP / 'admissions.csv.gz'}' AS a
    ON i.hadm_id = a.hadm_id
"""

stays = con.execute(query).df()
print(f"initial stays: {len(stays)}")
print(f"initial patients: {stays['subject_id'].nunique()}")

initial stays: 140
initial patients: 100


## Apply criteria step-by-step

In [10]:
def report(df, label):
    print(f"{label:<45} stays={len(df):>4}  patients={df['subject_id'].nunique():>4}")

report(stays, "initial")

# 1. Adults
cohort = stays[stays["anchor_age"] >= 18].copy()
report(cohort, "after adult filter (anchor_age >= 18)")

# 2. First ICU stay per patient
cohort = (
    cohort.sort_values(["subject_id", "intime"])
          .drop_duplicates(subset="subject_id", keep="first")
)
report(cohort, "after first ICU stay per patient")

# 3. LOS >= 24h (los is in days)
cohort = cohort[cohort["los"] >= 1.0].copy()
report(cohort, "after LOS >= 24h")

# 4. Drop rows with missing key fields
required = ["hospital_expire_flag", "gender", "anchor_age", "intime"]
cohort = cohort.dropna(subset=required).copy()
report(cohort, "after dropping missing key fields")

initial                                       stays= 140  patients= 100
after adult filter (anchor_age >= 18)         stays= 140  patients= 100
after first ICU stay per patient              stays= 100  patients= 100
after LOS >= 24h                              stays=  85  patients=  85
after dropping missing key fields             stays=  85  patients=  85


## Cohort summary

In [11]:
print(f"Final cohort size: {len(cohort)} stays / {cohort['subject_id'].nunique()} patients")
print(f"In-hospital mortality rate: {cohort['hospital_expire_flag'].mean():.1%}")
print()
print("Gender:")
print(cohort["gender"].value_counts())
print()
print("Insurance:")
print(cohort["insurance"].value_counts())

Final cohort size: 85 stays / 85 patients
In-hospital mortality rate: 8.2%

Gender:
gender
M    46
F    39
Name: count, dtype: int64

Insurance:
insurance
Other       48
Medicare    31
Medicaid     6
Name: count, dtype: int64


In [12]:
cohort.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
subject_id,85.0,NaN,NaN,NaN,10018723.705882,10001217.0,10008454.0,10019003.0,10024043.0,10040025.0,11415.07791
hadm_id,85.0,NaN,NaN,NaN,25133432.894118,20044587.0,22585261.0,24982426.0,28019404.0,29974575.0,3071279.380137
stay_id,85.0,NaN,NaN,NaN,34935083.447059,30057454.0,32374504.0,35009126.0,37127068.0,39880770.0,2857992.832309
intime,85,NaN,NaN,NaN,2149-02-15 16:31:09.411765,2110-04-11 15:52:22,2130-10-27 12:06:00,2146-06-22 11:46:29,2169-01-15 04:56:00,2201-10-30 12:25:00,NaN
outtime,85,NaN,NaN,NaN,2149-02-19 22:55:32.588235,2110-04-12 23:59:56,2130-10-29 12:05:02,2146-07-13 00:27:47,2169-01-20 15:47:50,2201-11-12 18:37:10,NaN
los,85.0,NaN,NaN,NaN,4.266935,1.04037,1.424757,2.688322,5.087512,20.528681,3.963623
first_careunit,85,8,Cardiac Vascular Intensive Care Unit (CVICU),24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,85,2,M,46,NaN,NaN,NaN,NaN,NaN,NaN,NaN
anchor_age,85.0,NaN,NaN,NaN,61.305882,21.0,50.0,63.0,72.0,91.0,16.832525
race,85,12,WHITE,52,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Save the cohort

Saving to `data/derived/cohort.parquet` — gitignored like the rest of `data/`. The feature and model notebooks load this file as their starting point.

In [14]:
out_path = DERIVED / "cohort.csv"
cohort.to_csv(out_path, index=False)
print(f"saved to {out_path}")
print(f"file size: {out_path.stat().st_size / 1024:.1f} KB")

saved to ../data/derived/cohort.csv
file size: 12.9 KB


## Takeaways

- The cohort is small after the LOS >= 24h filter — expected given the demo's size.
- Mortality rate is worth comparing with the raw rate from notebook 02 to confirm the filtering didn't introduce a major selection effect on the outcome.
- The cohort file is the single source of truth for the next steps. Anything downstream that needs demographics or the outcome reads from this file.

Next (notebook 04): extract first-24h features — vitals from `chartevents`, key labs from `labevents` — and build the feature matrix.